# 07 — Super-resolução (Real-ESRGAN): com vs. sem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/projeto-plumas/blob/main/07_super_resolucao/07_super_resolucao.ipynb)

**Objetivo:** avaliar se ampliar os quadros melhora a detecção, sem criar erros, comparando com um controle justo.

**Escopo da professora:** seção 5 (manter original e ampliada vinculadas; comparar nas mesmas imagens; dados de teste independentes; inspecionar artefatos) e pergunta 5.

**Lê:** `02_anotacao_split/yolo_dataset/` (quadros e rótulos de **teste**) e os pesos do YOLO (03).

**Grava:** `frames_sr/`, `pares_sr.csv`, `metricas_sr.csv`, figura de artefatos.

**GPU:** sim (Ambiente de execução → Alterar tipo → GPU T4)


In [ ]:
# === Preparo (igual em todos os notebooks) ===
import os, sys, subprocess

REPO = "USUARIO/projeto-plumas"      # troque pelo seu (veja README)
REPO_DIR = "/content/projeto-plumas"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    try:                                   # repositório privado: secret GITHUB_TOKEN
        from google.colab import userdata
        _tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        _tok = None
    _url = f"https://{_tok}@github.com/{REPO}.git" if _tok else f"https://github.com/{REPO}.git"
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True)
    else:
        _r = subprocess.run(["git", "clone", "-q", _url, REPO_DIR], capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("Falha no clone. Repositório privado? Configure o secret GITHUB_TOKEN (veja README).")
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:                                      # execução local: sobe até a raiz do repo
    _p = os.getcwd()
    while not os.path.exists(os.path.join(_p, "config.yaml")) and os.path.dirname(_p) != _p:
        _p = os.path.dirname(_p)
    os.chdir(_p)
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from pathlib import Path
from plumas.config import load_config, stage_dir, set_seed, log_experiment
cfg = load_config()
set_seed(cfg["seed"])
STAGE = "07_super_resolucao"
D = stage_dir(cfg, STAGE)                  # pasta desta etapa no Drive
print("Etapa:", STAGE, "| dados em:", D)


## Por que um controle (bicúbico)?
Se a imagem ampliada for reduzida de novo para `imgsz` na entrada do YOLO, não há ganho nenhum. E aumentar o tamanho de entrada **sem** super-resolução
já muda o resultado. Por isso comparamos quatro condições nos **mesmos quadros de teste**:

| Condição | Entrada |
|---|---|
| `orig_640` | quadro original, `imgsz` do config |
| `orig_SR` | quadro original, `imgsz` grande (o YOLO só redimensiona) |
| `bicubic_SR` | quadro ampliado por interpolação bicúbica, `imgsz` grande |
| `esrgan_SR` | quadro ampliado pelo Real-ESRGAN, `imgsz` grande |

Só `esrgan_SR` melhor que `bicubic_SR` indica ganho da super-resolução em si. Os rótulos YOLO são normalizados, então valem para imagens ampliadas.

In [ ]:
import subprocess, sys
# Real-ESRGAN tem dependências próprias. Pode haver conflito de versão (basicsr x torchvision);
# se o import falhar, veja o README do Real-ESRGAN (xinntao/Real-ESRGAN) para o ajuste necessário.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "realesrgan", "basicsr"], check=True)
import torch; print("GPU:", torch.cuda.is_available())

In [ ]:
import cv2, numpy as np, pandas as pd, shutil, yaml
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

SCALE = cfg["super_resolution"]["scale"]
WEIGHTS = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
up = RealESRGANer(scale=4, model_path=WEIGHTS, model=net, tile=0, half=torch.cuda.is_available())

D2 = stage_dir(cfg, "02_anotacao_split") / "yolo_dataset"
test_imgs = sorted((D2 / "images" / "test").glob("*.*"))
conds = {"bicubic": D / "frames_bicubic", "esrgan": D / "frames_sr"}
for p in conds.values():
    p.mkdir(exist_ok=True)

pairs = []
for p in test_imgs:
    img = cv2.imread(str(p))
    bic = cv2.resize(img, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_CUBIC)
    sr, _ = up.enhance(img, outscale=SCALE)
    cv2.imwrite(str(conds["bicubic"] / (p.stem + ".png")), bic)
    cv2.imwrite(str(conds["esrgan"] / (p.stem + ".png")), sr)
    pairs.append({"original": str(p), "bicubic": str(conds["bicubic"] / (p.stem + ".png")),
                  "esrgan": str(conds["esrgan"] / (p.stem + ".png")), "scale": SCALE})
pd.DataFrame(pairs).to_csv(D / "pares_sr.csv", index=False)      # vínculo original ↔ ampliada
print(len(pairs), "quadros ampliados")

## Inspecionar artefatos
Compare recortes da região da pluma: o Real-ESRGAN pode inventar bordas e texturas que não existem. Anote o que ver.

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(3, 3, figsize=(15, 12))
for row, pr in zip(axs, pairs[:: max(1, len(pairs) // 3)][:3]):
    o = cv2.imread(pr["original"]); b = cv2.imread(pr["bicubic"]); s = cv2.imread(pr["esrgan"])
    H, W = s.shape[:2]; y0, y1, x0, x1 = H // 4, H // 2, W // 4, W // 2           # recorte central (ajuste à pluma)
    for ax, im, tt in zip(row, [cv2.resize(o, (W, H), interpolation=cv2.INTER_NEAREST), b, s], ["original (pixels)", "bicúbico", "Real-ESRGAN"]):
        ax.imshow(cv2.cvtColor(im[y0:y1, x0:x1], cv2.COLOR_BGR2RGB)); ax.set_title(tt); ax.axis("off")
plt.tight_layout(); plt.savefig(D / "artefatos_sr.png", dpi=150); plt.show()

## Avaliar as quatro condições no teste
Monta um dataset YOLO com os mesmos rótulos para cada condição e roda a validação.

In [ ]:
from ultralytics import YOLO

BEST = Path("")                                  # best.pt escolhido na etapa 03
IMGSZ_SR = 1280                                  # entrada grande para as condições *_SR
labels_test = D2 / "labels" / "test"

def montar(nome, img_paths):
    root = D / f"yolo_{nome}"
    if root.exists():
        shutil.rmtree(root)
    for sub in ("images", "labels"):
        (root / sub / "test").mkdir(parents=True)
    for p in img_paths:
        shutil.copy2(p, root / "images" / "test" / Path(p).name)
        shutil.copy2(labels_test / (Path(p).stem + ".txt"), root / "labels" / "test" / (Path(p).stem + ".txt"))
    y = {"path": str(root), "train": "images/test", "val": "images/test", "test": "images/test", "names": {0: cfg["class_names"][0]}}
    (root / "data.yaml").write_text(yaml.safe_dump(y))       # train/val apontam para o teste só para o validador carregar
    return str(root / "data.yaml")

sets = {
    "orig_640": (montar("orig", test_imgs), cfg["yolo"]["imgsz"]),
    "orig_SR":  (montar("orig", test_imgs), IMGSZ_SR),
    "bicubic_SR": (montar("bicubic", [pr["bicubic"] for pr in pairs]), IMGSZ_SR),
    "esrgan_SR":  (montar("esrgan", [pr["esrgan"] for pr in pairs]), IMGSZ_SR),
}
m = YOLO(str(BEST))
rows = []
for nome, (data, sz) in sets.items():
    r = m.val(data=data, split="test", imgsz=sz, conf=0.001, iou=0.6, verbose=False, plots=False)
    rows.append({"condicao": nome, "imgsz": sz, "precision": float(r.box.mp), "recall": float(r.box.mr),
                 "mAP50": float(r.box.map50), "mAP50_95": float(r.box.map)})
res = pd.DataFrame(rows); res.to_csv(D / "metricas_sr.csv", index=False); res

## Segmentação com e sem SR
Repita a etapa 04 sobre os quadros de `frames_sr/` e compare IoU/Dice com as referências (`mascaras_gt`, reduzidas ou ampliadas à mesma escala). Anote aqui:

- A SR melhorou a segmentação? Em que condições (pluma pequena, baixo contraste)?
- Surgiram bordas artificiais que mudaram as máscaras?

In [ ]:
log_experiment(cfg, STAGE, {"modelo_sr": "RealESRGAN_x4plus", "escala": SCALE, "imgsz_sr": IMGSZ_SR,
                            "pesos_yolo": str(BEST), "resultado": res.to_dict(orient="records"),
                            "decisao": "SR avaliada contra controle bicúbico nos mesmos quadros de teste"})